# Notebook 2 – Adult Income (Binary Classification)

## Overview
We benchmark six tabular-learning methods on the UCI Adult Income dataset:
**TabNet**, **FT-Transformer**, **XGBoost**, **LightGBM**, **Random Forest**,
and **Logistic Regression**.  
Each model is tuned with **Optuna** (20 trials) and evaluated across 3 seeds.  
Metrics: **Accuracy**, **AUC-ROC**, **F1**.


In [ ]:
!pip install pytorch-tabnet "rtdl==0.0.13" optuna xgboost lightgbm ucimlrepo scikit-learn pandas numpy matplotlib seaborn shap

## Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import rtdl
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

from pytorch_tabnet.tab_model import TabNetClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## Configuration

In [ ]:
SEEDS           = [42, 123, 456]
N_OPTUNA_TRIALS = 20
TEST_SIZE       = 0.20
VAL_FRAC        = 0.25


## Data Loading & EDA

In [ ]:
adult = fetch_ucirepo(id=2)
X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

print("Features shape:", X_raw.shape)
print()
print(X_raw.dtypes)
print()
print(X_raw.describe())


In [ ]:
# Target distribution
target_series = y_raw.iloc[:, 0].astype(str).str.strip()
print("Unique target values:", target_series.unique())
print(target_series.value_counts())


In [ ]:
# Missing value overview
print("Missing values (including '?'):")
for col in X_raw.columns:
    n = (X_raw[col].astype(str).str.strip() == '?').sum()
    if n > 0:
        print(f"  {col}: {n}")


In [ ]:
# Distribution plots for numeric columns
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(X_raw[col].dropna(), bins=40, edgecolor='k', alpha=0.7)
    ax.set_title(col)
plt.suptitle("Adult Income – numerical feature distributions", fontsize=13)
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain',
            'capital-loss', 'hours-per-week']
cat_cols = ['workclass', 'education', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country']

# Binarize target
y_series = y_raw.iloc[:, 0].astype(str).str.strip()
y = np.where(y_series.str.contains('>50K'), 1, 0).astype(np.int64)
print(f"Positive rate: {y.mean():.3%}")

# Replace '?' with NaN
X_proc = X_raw.copy()
for col in cat_cols:
    X_proc[col] = X_proc[col].astype(str).str.strip().replace('?', np.nan)

# Impute
for col in num_cols:
    med = X_proc[col].median()
    X_proc[col] = X_proc[col].fillna(med)
for col in cat_cols:
    mode_val = X_proc[col].mode()[0]
    X_proc[col] = X_proc[col].fillna(mode_val)

print("Missing after imputation:", X_proc.isnull().sum().sum())


In [ ]:
# Encode
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat_enc = ord_enc.fit_transform(X_proc[cat_cols]).astype(np.int64)

scaler_num = StandardScaler()
X_num_enc  = scaler_num.fit_transform(X_proc[num_cols].values.astype(np.float32))

cat_cardinalities = [len(cats) for cats in ord_enc.categories_]
print("Cat cardinalities:", cat_cardinalities)
print("X_num shape:", X_num_enc.shape, "  X_cat shape:", X_cat_enc.shape)


## Data Splitting (60 / 20 / 20)

In [ ]:
# Combined array for sklearn models
X_all = np.concatenate([X_num_enc, X_cat_enc.astype(np.float32)], axis=1)

idx = np.arange(len(y))
idx_tv, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=42, stratify=y)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_FRAC, random_state=42,
                                       stratify=y[idx_tv])

X_train_all, X_val_all, X_test_all = X_all[idx_train], X_all[idx_val], X_all[idx_test]
X_train_num, X_val_num, X_test_num = X_num_enc[idx_train], X_num_enc[idx_val], X_num_enc[idx_test]
X_train_cat, X_val_cat, X_test_cat = X_cat_enc[idx_train], X_cat_enc[idx_val], X_cat_enc[idx_test]
y_train_clf, y_val_clf, y_test_clf  = y[idx_train], y[idx_val], y[idx_test]

# StandardScaler on combined (for TabNet)
sc2 = StandardScaler()
X_train_sc = sc2.fit_transform(X_train_all)
X_val_sc   = sc2.transform(X_val_all)
X_test_sc  = sc2.transform(X_test_all)

print(f"Train: {X_train_sc.shape}, Val: {X_val_sc.shape}, Test: {X_test_sc.shape}")


## Helper Functions

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_classification_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    return acc, auc, f1


def train_ft_transformer(model, X_num_tr, X_cat_tr, y_tr,
                          X_num_v, X_cat_v, y_v,
                          lr=1e-3, n_epochs=100, batch_size=256,
                          task='regression', device_='cpu'):
    model = model.to(device_)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss() if task == 'regression' else nn.BCEWithLogitsLoss()

    X_num_tr_t = torch.FloatTensor(X_num_tr).to(device_)
    X_cat_tr_t = torch.LongTensor(X_cat_tr).to(device_) if X_cat_tr is not None else None
    y_tr_t     = torch.FloatTensor(y_tr.astype(np.float32)).to(device_)
    X_num_v_t  = torch.FloatTensor(X_num_v).to(device_)
    X_cat_v_t  = torch.LongTensor(X_cat_v).to(device_) if X_cat_v is not None else None
    y_v_t      = torch.FloatTensor(y_v.astype(np.float32)).to(device_)

    train_losses, val_losses = [], []
    best_val   = float('inf')
    best_state = None
    patience   = 20
    pat_cnt    = 0

    for epoch in range(n_epochs):
        model.train()
        n   = len(X_num_tr_t)
        idx = torch.randperm(n)
        ep_loss = 0.0
        for i in range(0, n, batch_size):
            b  = idx[i:i+batch_size]
            xn = X_num_tr_t[b]
            xc = X_cat_tr_t[b] if X_cat_tr_t is not None else None
            yb = y_tr_t[b]
            optimizer.zero_grad()
            out  = model(xn, xc).squeeze(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            vout  = model(X_num_v_t, X_cat_v_t).squeeze(-1)
            vloss = criterion(vout, y_v_t).item()
        train_losses.append(ep_loss / n)
        val_losses.append(vloss)
        if vloss < best_val:
            best_val   = vloss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt    = 0
        else:
            pat_cnt += 1
        if pat_cnt >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses


def predict_ft_transformer(model, X_num, X_cat, device_, batch_size=512):
    model.eval()
    model   = model.to(device_)
    X_num_t = torch.FloatTensor(X_num).to(device_)
    X_cat_t = torch.LongTensor(X_cat).to(device_) if X_cat is not None else None
    preds   = []
    with torch.no_grad():
        for i in range(0, len(X_num_t), batch_size):
            xn  = X_num_t[i:i+batch_size]
            xc  = X_cat_t[i:i+batch_size] if X_cat_t is not None else None
            out = model(xn, xc).squeeze(-1)
            preds.append(out.cpu().numpy())
    return np.concatenate(preds)


## Model 1: TabNet

In [ ]:
all_results = []

def tabnet_clf_objective(trial):
    n_d     = trial.suggest_int('n_d', 8, 64)
    n_a     = trial.suggest_int('n_a', 8, 64)
    n_steps = trial.suggest_int('n_steps', 3, 10)
    gamma   = trial.suggest_float('gamma', 1.0, 2.0)
    lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    set_seed(42)
    m = TabNetClassifier(n_d=n_d, n_a=n_a, n_steps=n_steps, gamma=gamma,
                         optimizer_params={'lr': lr}, verbose=0, seed=42,
                         device_name='cuda' if torch.cuda.is_available() else 'cpu')
    m.fit(X_train_sc, y_train_clf.astype(int),
          eval_set=[(X_val_sc, y_val_clf.astype(int))],
          eval_name=['val'], eval_metric=['auc'],
          patience=15, max_epochs=100, batch_size=1024, virtual_batch_size=256)
    prob = m.predict_proba(X_val_sc)[:, 1]
    return -roc_auc_score(y_val_clf, prob)

study_tn = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_tn.optimize(tabnet_clf_objective, n_trials=N_OPTUNA_TRIALS)
best_tn = study_tn.best_params
print(f"Best TabNet params: {best_tn}")


In [ ]:
print("Training TabNet across seeds...")
for seed in SEEDS:
    set_seed(seed)
    m = TabNetClassifier(
        n_d=best_tn['n_d'], n_a=best_tn['n_a'], n_steps=best_tn['n_steps'],
        gamma=best_tn['gamma'], optimizer_params={'lr': best_tn['lr']},
        verbose=0, seed=seed,
        device_name='cuda' if torch.cuda.is_available() else 'cpu')
    m.fit(X_train_sc, y_train_clf.astype(int),
          eval_set=[(X_val_sc, y_val_clf.astype(int))],
          eval_name=['val'], eval_metric=['auc'],
          patience=20, max_epochs=200, batch_size=1024, virtual_batch_size=256)
    preds = m.predict(X_test_sc)
    probs = m.predict_proba(X_test_sc)[:, 1]
    acc, auc, f1 = compute_classification_metrics(y_test_clf, preds, probs)
    all_results.append({'method': 'TabNet', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Model 2: FT-Transformer

In [ ]:
n_num_ft = X_train_num.shape[1]

def ft_clf_objective(trial):
    d_token   = trial.suggest_categorical('d_token', [64, 128, 192])
    n_blocks  = trial.suggest_int('n_blocks', 1, 3)
    attn_drop = trial.suggest_float('attention_dropout', 0.0, 0.3)
    ffn_drop  = trial.suggest_float('ffn_dropout', 0.0, 0.3)
    lr        = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    set_seed(42)
    model = rtdl.FTTransformer.make_baseline(
        n_num_features=n_num_ft,
        cat_cardinalities=cat_cardinalities,
        d_token=d_token,
        n_blocks=n_blocks,
        attention_dropout=attn_drop,
        ffn_d_hidden=int(d_token * 4 / 3),
        ffn_dropout=ffn_drop,
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, _, _ = train_ft_transformer(
        model, X_train_num, X_train_cat, y_train_clf,
        X_val_num, X_val_cat, y_val_clf,
        lr=lr, n_epochs=50, batch_size=256,
        task='classification', device_=str(device))
    raw = predict_ft_transformer(model, X_val_num, X_val_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    return -roc_auc_score(y_val_clf, prob)

study_ft = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_ft.optimize(ft_clf_objective, n_trials=N_OPTUNA_TRIALS)
best_ft = study_ft.best_params
print(f"Best FT-Transformer params: {best_ft}")


In [ ]:
print("Training FT-Transformer across seeds...")
ft_train_curves = {}
for seed in SEEDS:
    set_seed(seed)
    model = rtdl.FTTransformer.make_baseline(
        n_num_features=n_num_ft,
        cat_cardinalities=cat_cardinalities,
        d_token=best_ft['d_token'],
        n_blocks=best_ft['n_blocks'],
        attention_dropout=best_ft['attention_dropout'],
        ffn_d_hidden=int(best_ft['d_token'] * 4 / 3),
        ffn_dropout=best_ft['ffn_dropout'],
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, tr_l, va_l = train_ft_transformer(
        model, X_train_num, X_train_cat, y_train_clf,
        X_val_num, X_val_cat, y_val_clf,
        lr=best_ft['lr'], n_epochs=100, batch_size=256,
        task='classification', device_=str(device))
    ft_train_curves[seed] = (tr_l, va_l)
    raw  = predict_ft_transformer(model, X_test_num, X_test_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    pred = (prob >= 0.5).astype(int)
    acc, auc, f1 = compute_classification_metrics(y_test_clf, pred, prob)
    all_results.append({'method': 'FT-Transformer', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Model 3: XGBoost

In [ ]:
def xgb_clf_objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'max_depth':     trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'random_state':  42, 'use_label_encoder': False, 'eval_metric': 'auc'
    }
    set_seed(42)
    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(X_train_all, y_train_clf)
    prob = m.predict_proba(X_val_all)[:, 1]
    return -roc_auc_score(y_val_clf, prob)

study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_clf_objective, n_trials=N_OPTUNA_TRIALS)
best_xgb = study_xgb.best_params
print(f"Best XGBoost params: {best_xgb}")


In [ ]:
print("Training XGBoost across seeds...")
xgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = xgb.XGBClassifier(**best_xgb, random_state=seed, verbosity=0,
                           use_label_encoder=False, eval_metric='auc')
    m.fit(X_train_all, y_train_clf)
    preds = m.predict(X_test_all)
    probs = m.predict_proba(X_test_all)[:, 1]
    acc, auc, f1 = compute_classification_metrics(y_test_clf, preds, probs)
    all_results.append({'method': 'XGBoost', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    xgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Model 4: LightGBM

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'num_leaves':    trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_state':  42, 'verbose': -1
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_train_all, y_train_clf)
    prob = m.predict_proba(X_val_all)[:, 1]
    return -roc_auc_score(y_val_clf, prob)

study_lgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_objective, n_trials=N_OPTUNA_TRIALS)
best_lgb = study_lgb.best_params
print(f"Best LightGBM params: {best_lgb}")


In [ ]:
print("Training LightGBM across seeds...")
lgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = lgb.LGBMClassifier(**best_lgb, random_state=seed, verbose=-1)
    m.fit(X_train_all, y_train_clf)
    preds = m.predict(X_test_all)
    probs = m.predict_proba(X_test_all)[:, 1]
    acc, auc, f1 = compute_classification_metrics(y_test_clf, preds, probs)
    all_results.append({'method': 'LightGBM', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    lgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Model 5: Random Forest

In [ ]:
def rf_clf_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state':    42
    }
    set_seed(42)
    m = RandomForestClassifier(**params, n_jobs=-1)
    m.fit(X_train_all, y_train_clf)
    prob = m.predict_proba(X_val_all)[:, 1]
    return -roc_auc_score(y_val_clf, prob)

study_rf = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(rf_clf_objective, n_trials=N_OPTUNA_TRIALS)
best_rf = study_rf.best_params
print(f"Best RF params: {best_rf}")


In [ ]:
print("Training Random Forest across seeds...")
rf_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = RandomForestClassifier(**best_rf, random_state=seed, n_jobs=-1)
    m.fit(X_train_all, y_train_clf)
    preds = m.predict(X_test_all)
    probs = m.predict_proba(X_test_all)[:, 1]
    acc, auc, f1 = compute_classification_metrics(y_test_clf, preds, probs)
    all_results.append({'method': 'RandomForest', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    rf_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Model 6: Logistic Regression

In [ ]:
def lr_objective(trial):
    C = trial.suggest_float('C', 0.01, 10.0, log=True)
    m = LogisticRegression(C=C, max_iter=1000, random_state=42)
    m.fit(X_train_all, y_train_clf)
    prob = m.predict_proba(X_val_all)[:, 1]
    return -roc_auc_score(y_val_clf, prob)

study_lr = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_lr.optimize(lr_objective, n_trials=N_OPTUNA_TRIALS)
best_lr = study_lr.best_params
print(f"Best LR params: {best_lr}")


In [ ]:
print("Training Logistic Regression across seeds...")
for seed in SEEDS:
    set_seed(seed)
    m = LogisticRegression(C=best_lr['C'], max_iter=1000, random_state=seed)
    m.fit(X_train_all, y_train_clf)
    preds = m.predict(X_test_all)
    probs = m.predict_proba(X_test_all)[:, 1]
    acc, auc, f1 = compute_classification_metrics(y_test_clf, preds, probs)
    all_results.append({'method': 'LogisticRegression', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, F1={f1:.4f}")


## Results

In [ ]:
df_res = pd.DataFrame(all_results)
summary = df_res.groupby('method').agg(
    acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
    auc_mean=('auc', 'mean'),      auc_std=('auc', 'std'),
    f1_mean=('f1', 'mean'),        f1_std=('f1', 'std')
).round(4)

summary['Accuracy'] = summary['acc_mean'].astype(str) + ' +/- ' + summary['acc_std'].astype(str)
summary['AUC-ROC']  = summary['auc_mean'].astype(str) + ' +/- ' + summary['auc_std'].astype(str)
summary['F1']       = summary['f1_mean'].astype(str)  + ' +/- ' + summary['f1_std'].astype(str)
print(summary[['Accuracy', 'AUC-ROC', 'F1']])


## Visualizations

In [ ]:
methods = summary.index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(methods, summary['acc_mean'].values, yerr=summary['acc_std'].values,
            capsize=5, color='steelblue', alpha=0.8)
axes[0].set_title('Accuracy (higher is better)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(methods, summary['auc_mean'].values, yerr=summary['auc_std'].values,
            capsize=5, color='darkorange', alpha=0.8)
axes[1].set_title('AUC-ROC (higher is better)')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(methods, summary['f1_mean'].values, yerr=summary['f1_std'].values,
            capsize=5, color='forestgreen', alpha=0.8)
axes[2].set_title('F1 (higher is better)')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Adult Income – Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# FT-Transformer training curves
fig, axes = plt.subplots(1, len(SEEDS), figsize=(15, 4))
for ax, seed in zip(axes, SEEDS):
    tr_l, va_l = ft_train_curves[seed]
    ax.plot(tr_l, label='Train')
    ax.plot(va_l, label='Val')
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend()
plt.suptitle('FT-Transformer Training Curves', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# LightGBM feature importance
if lgb_model_last is not None:
    all_cols = num_cols + cat_cols
    fi = lgb_model_last.feature_importances_
    fi_df = pd.DataFrame({'feature': all_cols, 'importance': fi}).sort_values(
        'importance', ascending=True)
    fi_df.plot.barh(x='feature', y='importance', figsize=(8, 6),
                    legend=False, color='mediumpurple')
    plt.title('LightGBM Feature Importance')
    plt.tight_layout()
    plt.show()


## Analysis & Conclusions

### Summary
We compared six methods on the UCI Adult Income binary classification task.

- **LightGBM** and **XGBoost** typically achieve top AUC-ROC on this dataset
  thanks to their ability to handle mixed numerical/categorical features.
- **FT-Transformer** leverages embeddings for categorical features, which can
  yield competitive AUC with sufficient training.
- **Logistic Regression** provides a fast and interpretable baseline.
- **TabNet** and **Random Forest** round out the comparison.

### Observations
- The dataset is moderately imbalanced (~24% positive); AUC-ROC is the most
  reliable metric here.
- OrdinalEncoding + StandardScaling is a straightforward preprocessing pipeline
  that works across all methods.
- 3-seed evaluation reveals model stability under random initialization.

### Next Steps
- One-hot encoding for tree methods vs. ordinal for neural methods.
- Class-weight adjustment to improve F1 for the minority class.
- SHAP explanations for the best-performing model.
